# About the Notebook

Chapter 1 ended with a harness that worked. This notebook turns that working
system into a component map and asks: **how do the harness components cooperate to control execution?**

The connective tissue is state. The model only decides over the state it
receives. The harness determines which state exists, which becomes visible,
which can change, which changes persist, and which transition is allowed next.

This notebook therefore follows three steps:

1. identify the kinds of state the run touches;
2. expose the harness components that read, write, constrain, or verify that state;
3. remove one relationship at a time and observe what fails.

The ablations are evidence for the component map, not the organizing idea. The
organizing idea is the interaction between components inside one harness.

<font color="red" size="6">
<b>ATTENTION</b>
</font>

> ### This notebook is a teaching artifact, not a starting template
>
> Everything here is deliberately small enough to read in one sitting. The
> environment is a dictionary, the cache is a dictionary, the trace is a list,
> and there is no persistence, concurrency, or isolation anywhere.
>
> That is not an oversight, and it is not a first draft of something you should
> deploy. Each simplification hides a genuine engineering problem that later
> chapters take apart properly. Every section below carries an **In production**
> note naming what the simplification is costing you and where the real version
> is covered.
>
> The final section collects those notes into one checklist. If you take
> anything from this notebook into a real system, take the *shape*, named
> components with seams between them, and rebuild the insides.

## The interaction map


| Interaction | Harness components involved | Role within the harness |
|---|---|---|
| State exposure | Context, memory, orchestration | Determines which state is projected into the current model call and what remains outside it. |
| Verification and correction | Verification, observability, orchestration | Evaluates a proposal against evidence or constraints and can accept it, return corrective feedback, retry, recover, or terminate. |
| Execution flow | Orchestration, verification, governance | Determines what may happen next. Branches, loops, retries, and graphs structure transitions within the harness. |
| State lifecycle | Context, workflow state, operational state, durable state | Controls which state is transient, which is updated, which is discarded, and which is persisted. |
| Long-running execution | Orchestration, persistence, recovery, governance | Preserves enough workflow state to continue across model calls, process boundaries, interruptions, and approvals. |
| Multi-agent coordination | Orchestration, shared state, context, permissions | Controls which state each agent receives, which artifacts cross agent boundaries, who may update shared state, and what persists. |

This notebook concentrates on the first four rows.

## State is the connective tissue

Before naming the components, classify what the run has to carry.

- **Model context**:the state currently projected into a model call.
- **Workflow state**: where execution is now: current step, retry position,
  blocked proposal, pending approval, and next transition.
- **Operational state**:  the world the harness can observe or change: service
  health, logs, deployments, files, APIs, databases, or other external systems.
- **Durable state**: state intentionally persisted beyond the current process
  or model call: verified results, checkpoints, approvals, artifacts, or memory.


A harness doesn't just provide an environment around the model. Its
components decide how these forms of state are exposed, validated, changed,
recorded, and persisted.

That distinction becomes important later in the chapter: a loop is one way the
orchestrator revisits state after verification, and a graph is one way allowed
state transitions can be represented. They are execution structures inside the
harness, not automatically layers beside it.

## Setup

Only the standard library and Pydantic. No agent framework, because the point
is to see the mechanism rather than to configure one.

In [1]:
from __future__ import annotations

import json
import time
from dataclasses import dataclass, field
from hashlib import sha256
from typing import Any, Callable

from pydantic import BaseModel, Field, ValidationError

INCIDENT_TICKET = """
At 14:05 UTC, checkout errors increased from below 1% to 18%.
Customers report timeouts after submitting payment.
The checkout service was deployed at 13:52 UTC.
Please identify the likely cause and recommend the safest immediate response.
""".strip()

## Operational state: the world outside the run

The world the harness acts on: service health, logs, and deployment records.

`fresh_environment()` returns a **new copy** each time. That matters here
because the ablations run the same trajectory repeatedly, and one variant
mutating production state must not contaminate the next.

> **In production.** This dictionary stands in for observability platforms,
> log aggregators, and deployment APIs, systems that are remote, slow, rate
> limited, occasionally down, and eventually consistent. A real tool layer needs
> timeouts, retries with backoff, partial-failure handling, and a policy for
> what the agent does when a data source is simply unavailable. Chapter 6 covers
> the tool and protocol layer; Chapter 4 covers where those calls execute.

In [2]:
def fresh_environment() -> dict[str, Any]:
    """A new copy of the world, so one ablation cannot contaminate the next."""
    return {
        "service_status": {
            "checkout": {
                "error_rate_pct": 18.2,
                "p95_latency_ms": 8100,
                "db_pool_in_use": 20,
                "db_pool_size": 20,
                "db_pool_wait_ms": 6900,
                "release": "checkout-2026.07.29.3",
            }
        },
        "logs": {
            "checkout": [
                "14:04:58 ERROR acquire connection timeout after 5000ms",
                "14:04:59 ERROR acquire connection timeout after 5000ms",
                "14:05:01 WARN db pool saturated: 20/20 connections in use",
            ]
        },
        "deployments": {
            "checkout-2026.07.29.3": {
                "deployed_at": "13:52 UTC",
                "changes": {"DB_POOL_SIZE": {"from": 80, "to": 20}},   # the cause
                "database_migration": False,
                "rollback_available": True,
                "previous_release": "checkout-2026.07.24.1",
            }
        },
    }

The connection-pool change is the actual cause of the incident. Note where it
lives: not in the ticket, and not in any document a retriever could have
fetched. It exists only in the environment, which is why only a system that can
query the environment can find it.

## Tools, and the executor that is deliberately missing

Two collections decide what can happen. `TOOL_SCHEMAS` is what the model can
see and propose. The registry is what the harness can actually run.

`rollback_service` appears in the schemas and, in the baseline, has no executor
bound to it. Company policy requires incident-commander approval, and the
harness does not hold that approval, so there is nothing to call.

> **In production.** Binding tools to closures over a mutable dict is fine for a
> notebook and wrong for a system. Real tools need argument validation at the
> boundary, per-tool timeouts and cost accounting, structured errors the model
> can act on, and a way to version schemas without breaking running agents.
> Exposing an action with no executor is also a teaching simplification, the
> more honest production pattern is an executor gated on an approval token, which notebook `ch02_state_across_execution_boundaries.ipynb` builds.

In [3]:
def build_tools(env: dict[str, Any]) -> dict[str, Callable[..., dict[str, Any]]]:
    """Read-only tools bound to one environment instance."""

    def get_service_status(service: str) -> dict[str, Any]:
        return env["service_status"].get(
            service, {"error": f"Unknown service: {service}"}
        )

    def search_logs(service: str, query: str) -> dict[str, Any]:
        lines = env["logs"].get(service, [])
        matches = [line for line in lines if query.lower() in line.lower()]
        return {"service": service, "query": query, "matches": matches}

    def get_deployment(release: str) -> dict[str, Any]:
        return env["deployments"].get(
            release, {"error": f"Unknown release: {release}"}
        )

    return {
        "get_service_status": get_service_status,
        "search_logs": search_logs,
        "get_deployment": get_deployment,
    }


def build_rollback(env: dict[str, Any]) -> Callable[..., dict[str, Any]]:
    """A real, side-effecting executor. Used only by the governance ablation."""

    def rollback_service(service: str) -> dict[str, Any]:
        status = env["service_status"][service]
        current = status["release"]
        previous = env["deployments"][current]["previous_release"]
        status["release"] = previous
        status["error_rate_pct"] = 0.4
        status["db_pool_size"] = 80
        return {"service": service, "rolled_back_from": current,
                "now_running": previous}

    return rollback_service


SIDE_EFFECTING = {"rollback_service"}


def _fn(name: str, description: str, props: dict[str, str]) -> dict[str, Any]:
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": {k: {"type": v} for k, v in props.items()},
                "required": list(props),
                "additionalProperties": False,
            },
        },
    }


TOOL_SCHEMAS = [
    _fn("get_service_status", "Read health and resource signals for a service.",
        {"service": "string"}),
    _fn("search_logs", "Search recent service logs for a literal fragment.",
        {"service": "string", "query": "string"}),
    _fn("get_deployment", "Read metadata for a named release.",
        {"release": "string"}),
    _fn("rollback_service", "Roll a service back to its previous release.",
        {"service": "string"}),
]

## The result contract

What the harness will accept as an answer. Everything else is a proposal.

> **In production.** This schema is loose on purpose so the ablations stay
> legible. `confidence` accepts any string, which means "high" and "banana" both
> pass, a `Literal["low", "medium", "high"]` would bite harder. Real contracts
> also need versioning, because tightening a schema against agents already
> running in production is a migration, not an edit.

In [4]:
class IncidentPlan(BaseModel):
    likely_cause: str
    confidence: str
    immediate_actions: list[str] = Field(min_length=1)
    evidence: list[str] = Field(min_length=1)
    escalation_condition: str

## Component 1: Observability records what happened

The recorder holds what happened. Verification reads from it, which is the
relationship the first ablation cuts.

> **In production.** A list in memory is not observability. Use OpenTelemetry
> spans or an equivalent, emit to a collector rather than accumulating in the
> process, sample high-volume events, and redact payloads before they leave the
> box, traces of agent runs contain whatever the user typed. You'll see this in Chapter 8.

In [5]:
@dataclass
class TraceEvent:
    kind: str
    payload: dict[str, Any]
    timestamp: float = field(default_factory=time.time)


class Recorder:
    def __init__(self) -> None:
        self.events: list[TraceEvent] = []

    def record(self, kind: str, payload: dict[str, Any] | None = None) -> None:
        self.events.append(TraceEvent(kind, payload or {}))

    def observed_sources(self) -> set[str]:
        """Tools that actually produced an observation on this run."""
        return {e.payload["tool"] for e in self.events if e.kind == "tool_completed"}

    def kinds(self) -> list[str]:
        return [e.kind for e in self.events]

## Component 2: Governance constrains allowed transitions

Decides whether a proposal may become an action. Note that it decides on the
*tool name*, before anything runs.

> **In production.** Deciding on the tool name alone ignores arguments
> `delete(scope="all")` and `delete(scope="one_row")` are the same tool. Part 2
> replaces this with an authenticated principal, a role check, and a pending
> action carrying its own arguments and identity. Chapter 12.

In [6]:
@dataclass
class Decision:
    permitted: bool
    reason: str = ""


class ApprovalGovernor:
    """Side-effecting tools need an approval the harness can point to."""

    def __init__(self, approvals: set[str] | None = None) -> None:
        self.approvals = approvals or set()

    def decide(self, tool_name: str, arguments: dict[str, Any]) -> Decision:
        if tool_name not in SIDE_EFFECTING:
            return Decision(True)
        if tool_name in self.approvals:
            return Decision(True, "approved")
        return Decision(False,
                        "This action requires incident-commander approval, which "
                        "the harness does not hold. Recommend it instead of "
                        "performing it.")

## Component 3: Execution turns proposals into effects

The only path from a proposal to an effect. Every call passes governance first,
and every outcome is recorded.

> **In production.** This runs tools in the same process as the orchestrator,
> which means a tool can exhaust memory, block the loop, or read anything the
> process can read. Real execution belongs in a sandbox with resource bounds and
> its own filesystem and network policy. You'll see this in Chapter 4.

In [7]:
class ToolGateway:
    def __init__(self, registry: dict[str, Callable[..., dict[str, Any]]],
                 governor: Any, recorder: Recorder) -> None:
        self.registry = registry
        self.governor = governor
        self.recorder = recorder

    def call(self, tool_name: str, raw_arguments: str) -> dict[str, Any]:
        decision = self.governor.decide(tool_name, {})
        if not decision.permitted:
            self.recorder.record("tool_blocked", {"tool": tool_name})
            return {"error": f"Tool not permitted: {tool_name}",
                    "reason": decision.reason}

        if tool_name not in self.registry:
            self.recorder.record("tool_unavailable", {"tool": tool_name})
            return {"error": f"No executor bound for: {tool_name}"}

        try:
            arguments = json.loads(raw_arguments)
            self.recorder.record("tool_started",
                                 {"tool": tool_name, "arguments": arguments})
            result = self.registry[tool_name](**arguments)
            self.recorder.record("tool_completed",
                                 {"tool": tool_name, "result": result})
            return result
        except Exception as exc:
            self.recorder.record("tool_failed",
                                 {"tool": tool_name, "error": str(exc)})
            return {"error": str(exc)}

## Component 4: Verification reads evidence and state

Parses, validates against the contract, and grounds each evidence item against
what the recorder saw run. The last step is the one that depends on
observability, and it is what the first ablation removes.

> **In production.** Substring matching on source names is fragile: an evidence
> item mentioning a tool in passing will pass, and a correctly grounded item
> that names the source differently will fail. Ask the model for structured
> evidence with an explicit source field rather than parsing prose. Chapters 8
> and 12 cover verification properly.

In [8]:
def parse_json_object(text: str) -> dict[str, Any]:
    """Parse JSON, tolerating a fenced code block if the model emits one."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(cleaned.strip())


@dataclass
class VerificationOutcome:
    plan: IncidentPlan | None
    error: str | None = None

    @property
    def ok(self) -> bool:
        return self.plan is not None


class Verifier:
    def __init__(self, recorder: Recorder) -> None:
        self.recorder = recorder

    def allowed_sources(self) -> set[str]:
        """What evidence may cite: whatever actually ran, plus the ticket."""
        return self.recorder.observed_sources() | {"incident ticket"}

    def verify(self, content: str) -> VerificationOutcome:
        try:
            plan = IncidentPlan.model_validate(parse_json_object(content))
        except (json.JSONDecodeError, ValidationError) as exc:
            return VerificationOutcome(None, str(exc))

        allowed = self.allowed_sources()
        ungrounded = [item for item in plan.evidence
                      if not any(marker in item for marker in allowed)]
        if ungrounded:
            return VerificationOutcome(
                None,
                f"Evidence cites a source that produced no observation on this "
                f"run. Observed: {sorted(allowed)}. Offending: {ungrounded}")
        return VerificationOutcome(plan)

## Component 5: Adaptation reuses only accepted state

Reuse, but only of results that passed verification. The write happens *after*
the check, which is the relationship the third ablation cuts.

> **In production.** This cache is keyed on the request text alone, so it will
> serve a stale plan for an environment that has since changed, and an edit to
> the request orphans the run entirely. Part 2 replaces the key with a
> harness-owned run identity. Chapters 9 to 11 will cover this.

In [9]:
def content_signature(text: str) -> str:
    return sha256(" ".join(text.lower().split()).encode()).hexdigest()[:16]


class VerifiedCache:
    def __init__(self) -> None:
        self.store: dict[str, dict[str, Any]] = {}

    def get(self, key: str) -> dict[str, Any] | None:
        return self.store.get(key)

    def put_verified(self, key: str, plan: IncidentPlan) -> None:
        self.store[key] = plan.model_dump()

    def put_unverified(self, key: str, raw: dict[str, Any]) -> None:
        raise RuntimeError("This cache only accepts verified plans.")

## Component 6 — Orchestration moves execution through state

Orchestration decides which transition is attempted next. In this notebook it is
a bounded `for` loop. The same responsibility can be represented as an explicit
graph, a state machine, a queue-driven workflow, or another topology.

The important boundary is responsibility, not syntax: verification can send a
proposal back for correction, governance can block an effect, and tool results
can create new state for the next model call. The orchestrator coordinates those
transitions inside the harness. A loop or graph is therefore a structure through
which orchestration operates, not a peer layer beside the harness.

> **In production.** A `for` loop with a step ceiling is one orchestration
> topology; an explicit graph with declared transitions is another, and larger
> workflows usually want the second. Either way this loop holds all its state in
> local variables, so it cannot survive the process — which is exactly what
> notebook `ch02_state_across_execution_boundaries.ipynb` fixes. Chapter 7 will cover this in depth.

In [10]:
HARNESS_SYSTEM = """
You are the decision component inside a governed incident-response harness.

Inspect enough runtime evidence to identify the most likely cause of the incident.
You may call any tool the harness has made available to you. The harness, not you,
decides which of those calls actually execute. When the evidence supports a
remediation that a tool exposes, call that tool rather than describing it.

When you have enough evidence, return only a JSON object with:
- likely_cause
- confidence
- immediate_actions
- evidence
- escalation_condition

Evidence entries must explicitly name the source they came from, using the name
of the tool that produced the observation, or "incident ticket".
""".strip()


@dataclass
class RunResult:
    plan: IncidentPlan | None
    recorder: Recorder
    escalated: bool = False
    cache_hit: bool = False


class Orchestrator:
    def __init__(self, client: Any, model: str, gateway: ToolGateway,
                 verifier: Verifier, recorder: Recorder,
                 cache: VerifiedCache | None = None,
                 cache_before_verify: bool = False) -> None:
        self.client = client
        self.model = model
        self.gateway = gateway
        self.verifier = verifier
        self.recorder = recorder
        self.cache = cache
        self.cache_before_verify = cache_before_verify

    def run(self, ticket: str, max_steps: int = 8) -> RunResult:
        signature = content_signature(ticket)

        if self.cache is not None and self.cache.get(signature) is not None:
            self.recorder.record("cache_hit", {"signature": signature})
            return RunResult(IncidentPlan.model_validate(self.cache.get(signature)),
                             self.recorder, cache_hit=True)

        messages: list[dict[str, Any]] = [
            {"role": "system", "content": HARNESS_SYSTEM},
            {"role": "user", "content": ticket},
        ]

        for step in range(1, max_steps + 1):
            self.recorder.record("model_call_started", {"step": step})
            response = self.client.chat.completions.create(
                model=self.model, temperature=0, messages=messages,
                tools=TOOL_SCHEMAS, tool_choice="auto")
            message = response.choices[0].message
            self.recorder.record("model_call_completed", {
                "step": step,
                "finish_reason": response.choices[0].finish_reason,
                "usage": getattr(response, "usage", None),
            })

            assistant: dict[str, Any] = {"role": "assistant",
                                         "content": message.content}
            if message.tool_calls:
                assistant["tool_calls"] = [
                    {"id": tc.id, "type": "function",
                     "function": {"name": tc.function.name,
                                  "arguments": tc.function.arguments}}
                    for tc in message.tool_calls]
            messages.append(assistant)

            if message.tool_calls:
                for tool_call in message.tool_calls:
                    result = self.gateway.call(tool_call.function.name,
                                               tool_call.function.arguments)
                    messages.append({"role": "tool",
                                     "tool_call_id": tool_call.id,
                                     "content": json.dumps(result)})
                continue

            if self.cache_before_verify and self.cache is not None:
                try:
                    self.cache.put_unverified(
                        signature, parse_json_object(message.content))
                    self.recorder.record("cached_before_verify", {"step": step})
                except Exception:
                    pass

            outcome = self.verifier.verify(message.content or "")
            if outcome.ok:
                self.recorder.record("verification_passed", {"step": step})
                if self.cache is not None and not self.cache_before_verify:
                    self.cache.put_verified(signature, outcome.plan)
                    self.recorder.record("verified_plan_cached",
                                         {"signature": signature})
                return RunResult(outcome.plan, self.recorder)

            self.recorder.record("verification_failed",
                                 {"step": step, "error": outcome.error})
            messages.append({"role": "user", "content": (
                "The harness rejected the previous answer. Verification error: "
                f"{outcome.error}. Return a corrected JSON object using only "
                "observed evidence.")})

        self.recorder.record("escalated", {"reason": "step_budget_exhausted"})
        return RunResult(None, self.recorder, escalated=True)

## Why a recorded run rather than a live model

Every ablation below asks the same question: given the **same** model behaviour,
what does the surrounding system do differently? A live model answers a slightly
different question on each call, and the comparison stops meaning anything.

So the client below replays one recorded trajectory. `RECORDED_RUN` is the
Chapter 1 run: three evidence calls, a second deployment lookup, a
`rollback_service` proposal, a malformed answer, and a corrected one.

This is also how harness changes get evaluated in practice replaying recorded
trajectories against a modified harness is an offline evaluation loop, and
Chapters 8 and 12 build on it.

> **In production.** Two caveats. First, a replayed trajectory tells you what
> your harness does with *last week's* model behaviour, not this week's; it
> complements live evaluation rather than replacing it. Second, the fixture
> below is written out by hand so this notebook runs without an API key. Capture
> your own by wrapping a real client and appending each assistant turn, or the
> ablations will be testing our model's habits rather than yours.

In [11]:
@dataclass
class _Function:
    name: str
    arguments: str


@dataclass
class _ToolCall:
    id: str
    function: _Function
    type: str = "function"


@dataclass
class _Message:
    content: str | None = None
    tool_calls: list[_ToolCall] | None = None


@dataclass
class _Choice:
    message: _Message
    finish_reason: str


@dataclass
class _Response:
    choices: list[_Choice]
    usage: Any = None


class ReplayClient:
    """Yields recorded assistant turns in order, ignoring the messages sent."""

    def __init__(self, turns: list[dict[str, Any]]) -> None:
        self.turns = turns
        self.calls = 0
        self.chat = self          # so client.chat.completions.create resolves
        self.completions = self

    def create(self, **kwargs: Any) -> _Response:
        if self.calls >= len(self.turns):
            raise RuntimeError(
                "The replay ran out of recorded turns: the harness under test "
                "took more steps than the recording contains.")
        turn = self.turns[self.calls]
        self.calls += 1
        if "tool_calls" in turn:
            calls = [_ToolCall(f"call_{self.calls}_{i}",
                               _Function(tc["name"], json.dumps(tc["arguments"])))
                     for i, tc in enumerate(turn["tool_calls"])]
            return _Response([_Choice(_Message(None, calls), "tool_calls")])
        return _Response([_Choice(_Message(turn["content"], None), "stop")])

In [12]:
_GOOD_PLAN = {
    "likely_cause": "The checkout deployment at 13:52 UTC reduced DB_POOL_SIZE "
                    "from 80 to 20, exhausting the database connection pool.",
    "confidence": "Very high",
    "immediate_actions": [
        "Request approval to roll back checkout, since the rollback tool "
        "reported that approval is required.",
        "Monitor error rate and pool utilisation during and after rollback.",
    ],
    "evidence": [
        "Source get_service_status: 20 of 20 database connections are in use.",
        "Source search_logs: 'acquire connection timeout after 5000ms' at 14:04:58.",
        "Source get_deployment: release checkout-2026.07.29.3 changed "
        "DB_POOL_SIZE from 80 to 20.",
        "Source incident ticket: checkout errors rose to 18% at 14:05 UTC.",
    ],
    "escalation_condition": "Escalate if rollback cannot be approved or does not "
                            "reduce checkout errors.",
}

# The step-4 answer from the real run: confidence as a float, evidence as
# objects. Both violate the contract, and the harness sent it back for repair.
_MALFORMED_PLAN = {**_GOOD_PLAN, "confidence": 0.99, "evidence": [
    {"source": "get_service_status", "detail": "20 of 20 connections in use."},
    {"source": "get_deployment", "detail": "DB_POOL_SIZE 80 -> 20."},
]}

RECORDED_RUN = [
    {"tool_calls": [
        {"name": "get_service_status", "arguments": {"service": "checkout"}},
        {"name": "search_logs", "arguments": {"service": "checkout",
                                              "query": "timeout"}},
        {"name": "get_deployment", "arguments": {"release": "checkout"}},
    ]},
    {"tool_calls": [{"name": "get_deployment",
                     "arguments": {"release": "checkout-2026.07.29.3"}}]},
    {"tool_calls": [{"name": "rollback_service",
                     "arguments": {"service": "checkout"}}]},
    {"content": json.dumps(_MALFORMED_PLAN)},
    {"content": json.dumps(_GOOD_PLAN)},
]

# A second fixture: one evidence call, then a plan citing two sources.
# Models do this. The question is whether the harness notices.
_UNGROUNDED_PLAN = {**_GOOD_PLAN, "evidence": [
    "Source get_service_status: all 20 of 20 database connections in use.",
    "Source search_logs: repeated connection timeout errors before 14:05.",
]}

UNGROUNDED_RUN = [
    {"tool_calls": [{"name": "get_service_status",
                     "arguments": {"service": "checkout"}}]},
    {"content": json.dumps(_UNGROUNDED_PLAN)},
]

print("recorded turns:", len(RECORDED_RUN), "| ungrounded turns:", len(UNGROUNDED_RUN))

recorded turns: 5 | ungrounded turns: 2


## Assembling a harness

One function builds a harness from the six components. Each keyword swaps
exactly one of them, which is what makes an ablation a one-line change.

In [13]:
READ_ONLY_TOOLS = {"get_service_status", "search_logs", "get_deployment"}


def show_trace(recorder: Recorder, label: str = "TRACE") -> None:
    print(label)
    print("=" * len(label))
    if not recorder.events:
        print("  (nothing recorded)")
        return
    for event in recorder.events:
        payload = dict(event.payload)
        if event.kind == "tool_completed":
            payload = {"tool": payload["tool"]}
        elif event.kind == "model_call_completed":
            # token counts and cost stay on the event; filtered for readability
            payload = {"step": payload["step"],
                       "finish_reason": payload["finish_reason"]}
        elif event.kind == "verification_failed":
            payload = {"step": payload.get("step"),
                       "error": str(payload.get("error", ""))[:64] + " ..."}
        print(f"  {event.kind:24} {payload}")


def assemble(turns, *, governor=None, verifier=None, recorder=None,
             cache=None, cache_before_verify=False, bind_rollback=False):
    """Build a harness from components. Each keyword swaps exactly one edge."""
    env = fresh_environment()
    recorder = recorder or Recorder()

    registry = build_tools(env)
    if bind_rollback:
        registry["rollback_service"] = build_rollback(env)

    gateway = ToolGateway(registry, governor or ApprovalGovernor(), recorder)
    verifier = verifier(recorder) if callable(verifier) else Verifier(recorder)

    orchestrator = Orchestrator(ReplayClient(turns), "replay", gateway,
                                verifier, recorder, cache, cache_before_verify)
    return env, recorder, orchestrator

## The intact harness

The baseline. This reproduces the Chapter 1 trace event for event, because it is
the same harness with names attached. The refactor bought nothing yet that is
the honest thing to say about it, and the ablations are what it buys.

In [14]:
env, recorder, orchestrator = assemble(RECORDED_RUN, cache=VerifiedCache())
baseline = orchestrator.run(INCIDENT_TICKET)

show_trace(recorder, "BASELINE")
print()
print("plan verified:", baseline.plan is not None)
print("release unchanged:", env["service_status"]["checkout"]["release"])

BASELINE
  model_call_started       {'step': 1}
  model_call_completed     {'step': 1, 'finish_reason': 'tool_calls'}
  tool_started             {'tool': 'get_service_status', 'arguments': {'service': 'checkout'}}
  tool_completed           {'tool': 'get_service_status'}
  tool_started             {'tool': 'search_logs', 'arguments': {'service': 'checkout', 'query': 'timeout'}}
  tool_completed           {'tool': 'search_logs'}
  tool_started             {'tool': 'get_deployment', 'arguments': {'release': 'checkout'}}
  tool_completed           {'tool': 'get_deployment'}
  model_call_started       {'step': 2}
  model_call_completed     {'step': 2, 'finish_reason': 'tool_calls'}
  tool_started             {'tool': 'get_deployment', 'arguments': {'release': 'checkout-2026.07.29.3'}}
  tool_completed           {'tool': 'get_deployment'}
  model_call_started       {'step': 3}
  model_call_completed     {'step': 3, 'finish_reason': 'tool_calls'}
  tool_blocked             {'tool': 'rollback

## Audit the state after one intact run

Now return to the four state lifetimes introduced at the beginning and locate
them in the assembled harness.

- **Model context** is the `messages` list projected into each model call.
- **Operational state** is the environment: service health, logs, deployments,
  and the release actually running.
- **Durable state** is represented here by the verified cache.
- **Workflow state** is still hidden in local execution: `step`, retry position,
  blocked proposals, and what should happen next.

That last category is the deliberate gap. The harness can control workflow
state during this process, but it has nowhere durable to put it. A process death
therefore destroys the task's position even though other state may survive.

Notebook `ch02_state_across_execution_boundaries.ipynb` externalizes workflow state and then asks what happens when the run
crosses process, time, human, and agent boundaries.

In [15]:
for kind, where in {
    "model context": "messages list, rebuilt from scratch on every run",
    "operational":   "the environment: services, logs, deployments",
    "durable":       "VerifiedCache, keyed on a hash of the ticket text",
    "workflow":      "step counter, retry position, refused proposals",
}.items():
    print(f"{kind:16} {where}")

print()
print("survives the process:        ", ["durable"])
print("survives an edit to request: ", [])   # the cache key *is* the request

model context    messages list, rebuilt from scratch on every run
operational      the environment: services, logs, deployments
durable          VerifiedCache, keyed on a hash of the ticket text
workflow         step counter, retry position, refused proposals

survives the process:         ['durable']
survives an edit to request:  []


The second line is worth sitting with. Reuse is keyed on
`content_signature(ticket)`, a hash of the request text, so changing one word
changes the key and orphans everything the run has already done.

Content-derived keys cannot survive iteration. Notebook `ch02_state_across_execution_boundaries.ipynb` replaces this with a
run id the harness mints and owns.

In [16]:
edited = INCIDENT_TICKET.replace("safest immediate response", "safest response")
print("original:", content_signature(INCIDENT_TICKET))
print("edited:  ", content_signature(edited))
print("same task, different key:",
      content_signature(INCIDENT_TICKET) != content_signature(edited))

original: 8de7851d9fa44690
edited:   76d58cd2566c314d
same task, different key: True


## Ablations: test the relationships between components

The component map matters only if its edges have consequences. Each ablation
removes exactly one relationship and replays the same recorded model behavior.
The model is identical in every case, so each failure is caused by the harness
relationship that was removed.

This is the useful role of the ablations: they show that reliability does not
live in isolated boxes. It emerges from interactions such as observability
feeding verification, governance constraining execution, and verification
gating what adaptation is allowed to retain.

### Ablation 1: trace → verification

The intact verifier grounds evidence against tools that actually produced an
observation. Cut that edge and grounding checks the *declared* tool surface
instead — the registry keys, which are known before anything runs.

In [17]:
class RegistryVerifier(Verifier):
    """Ablation: grounds against the declared tool surface, not the trace."""

    def __init__(self, recorder: Recorder, registry_names: set[str]) -> None:
        super().__init__(recorder)
        self.registry_names = registry_names

    def allowed_sources(self) -> set[str]:
        return set(self.registry_names) | {"incident ticket"}


env, recorder, orchestrator = assemble(
    UNGROUNDED_RUN, verifier=lambda r: RegistryVerifier(r, READ_ONLY_TOOLS))
broken = orchestrator.run(INCIDENT_TICKET, max_steps=2)

print("tools that actually ran:", sorted(recorder.observed_sources()))
print("evidence cites:         ", ["get_service_status", "search_logs"])
print("accepted:", broken.plan is not None)

tools that actually ran: ['get_service_status']
evidence cites:          ['get_service_status', 'search_logs']
accepted: True


In [18]:
env, recorder, orchestrator = assemble(UNGROUNDED_RUN)
intact = orchestrator.run(INCIDENT_TICKET, max_steps=2)

print("accepted:", intact.plan is not None, "| escalated:", intact.escalated)
print()
print("Without the edge, a plan citing a log search that never happened is")
print("indistinguishable from one that did. The claim is fabricated and the")
print("system reports success.")

accepted: False | escalated: True

Without the edge, a plan citing a log search that never happened is
indistinguishable from one that did. The claim is fabricated and the
system reports success.


### Ablation 2: governance → tools

Here `rollback_service` gets a real executor, so the governor is the only thing
between a proposal and a production change. Replay the same recorded run,
including the model's actual step-3 rollback proposal against a permissive
governor.

In [19]:
class PermissiveGovernor:
    """Ablation: every proposal executes."""

    def decide(self, tool_name: str, arguments: dict[str, Any]) -> Decision:
        return Decision(True)


env, recorder, orchestrator = assemble(
    RECORDED_RUN, governor=PermissiveGovernor(), bind_rollback=True)
before = env["service_status"]["checkout"]["release"]
orchestrator.run(INCIDENT_TICKET)

print(f"release: {before}  ->  {env['service_status']['checkout']['release']}")
print("blocked events:", [k for k in recorder.kinds() if k == "tool_blocked"])

release: checkout-2026.07.29.3  ->  checkout-2026.07.24.1
blocked events: []


In [20]:
env, recorder, orchestrator = assemble(RECORDED_RUN, bind_rollback=True)
before = env["service_status"]["checkout"]["release"]
orchestrator.run(INCIDENT_TICKET)

print(f"release: {before}  ->  {env['service_status']['checkout']['release']}")
print("blocked events:", [k for k in recorder.kinds() if k == "tool_blocked"])
print()
print("Same proposal, same model. One unapproved production change, caused")
print("entirely by which component was in the path.")

release: checkout-2026.07.29.3  ->  checkout-2026.07.29.3
blocked events: ['tool_blocked']

Same proposal, same model. One unapproved production change, caused
entirely by which component was in the path.


### Ablation 3: verification → adaptation

The intact cache writes only after verification passes. Cut that edge and it
writes on the first final answer, before anyone has checked it.

Watch what this does to the run that *repaired itself*. The run still succeeds.
The cache does not.

In [21]:
class EagerCache(VerifiedCache):
    """Ablation: cache first, check later. First write wins, as caches do."""

    def put_unverified(self, key: str, raw: dict[str, Any]) -> None:
        self.store.setdefault(key, raw)


cache = EagerCache()
env, recorder, orchestrator = assemble(RECORDED_RUN, cache=cache,
                                       cache_before_verify=True)
result = orchestrator.run(INCIDENT_TICKET)

print("this run repaired and passed:", result.plan is not None)

stored = cache.get(content_signature(INCIDENT_TICKET))
print("cached confidence:", repr(stored["confidence"]))
try:
    IncidentPlan.model_validate(stored)
    print("cached object is valid: True")
except ValidationError as exc:
    print("cached object is valid: False")
    print(" ", str(exc).splitlines()[0])

this run repaired and passed: True
cached confidence: 0.99
cached object is valid: False
  3 validation errors for IncidentPlan


The run succeeded and the cache is poisoned. The step-4 answer was written
first, first write wins, and the corrected step-5 answer never replaced it.

Every later request with this signature receives an object the verifier would
have rejected — and receives it *without* a model call, so nothing downstream
gets another chance to notice.

This is the ablation most likely to reach production unnoticed, because the run
that creates the problem reports success.

### Ablation 4: observability → everything

Remove the recorder. Nothing is retained, so grounding has nothing to check
against.

In [22]:
class NullRecorder(Recorder):
    """Ablation: observability removed. Nothing is retained."""

    def record(self, kind: str, payload: dict[str, Any] | None = None) -> None:
        return


env, recorder, orchestrator = assemble(RECORDED_RUN, recorder=NullRecorder())
blind = orchestrator.run(INCIDENT_TICKET, max_steps=5)

print("events recorded:", len(recorder.events))
print("plan accepted:", blind.plan is not None, "| escalated:", blind.escalated)
print()
print("Verification cannot pass, because the set of sources it is allowed to")
print("trust is empty. Observability is not a reporting feature here. It is an")
print("input to a decision.")

events recorded: 0
plan accepted: False | escalated: True

Verification cannot pass, because the set of sources it is allowed to
trust is empty. Observability is not a reporting feature here. It is an
input to a decision.


## What the ablations showed

| Edge removed | Failure |
|---|---|
| trace → verification | Fabricated evidence accepted; run reports success |
| governance → tools | Unapproved production change from the same proposal |
| verification → adaptation | Poisoned cache, created by the successful run |
| observability → everything | Nothing to ground against, so nothing passes |

Three of those four failures are silent. Only the last announces itself.

That is the argument for treating the harness as a substrate rather than as
plumbing: these relationships are load bearing, they fail quietly, and you can
only test a relationship that exists as a named edge.

## From this notebook to a production harness

Collected from the notes above, in the order they will bite.

| Here | In production | Chapter |
|---|---|---|
| Tools mutate an in-process dict | Remote systems with timeouts, retries, partial failure | 6 |
| Tools run in the orchestrator process | Sandbox with resource bounds and its own network policy | 4 |
| Permission is a set membership test on the tool name | Authenticated principal and role check on a durable pending action | Part 2, 12 |
| Trace is a list in memory | Spans emitted to a collector, sampled and redacted | 8 |
| Grounding is substring matching on prose | Structured evidence with an explicit source field | 8, 12 |
| Cache is unbounded and keyed on the request text | Harness-owned run identity, plus a key covering what determines the answer | Part 2, 9–11 |
| `confidence` accepts any string | Constrained enum, and a plan for schema versioning | 12 |
| Orchestration is a `for` loop with a ceiling | A topology chosen for the workflow, loop or graph | 7 |
| All workflow state is local variables | Transactional checkpoint store, resumable across processes | Part 2 |

Three of those rows are answered directly in Part 2, which is where the run
stops being a thing that lives inside one Python process. The rest belong to
the chapters named beside them.

